In [3]:
import numpy as np
from numba import njit

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import scipy.sparse as sp
from scipy.spatial.distance import directed_hausdorff
import scipy.linalg as la

from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error

import optuna
import optuna.visualization as vis
from optuna.importance import PedAnovaImportanceEvaluator

import warnings

In [4]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

# Init

In [5]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [6]:
@njit
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [7]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [8]:
henon_scaler = StandardScaler()
henon_scaled = henon_scaler.fit_transform(henon_dataset)

In [9]:
def henon_plot(data_list, names=None):
    fig = go.Figure()
    colors = ["white", "magenta"]

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scatter(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )

    fig.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
    )

    return fig

In [10]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(go.Scatter(
            x=actual, y=predicted, mode="markers",
            name="Data", marker=dict(color="rgba(50, 50, 200, 0.5)", size=5)
        ), row=1, col=col)

        min_val, max_val = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val], mode="lines", 
            name="Ideal", line=dict(color="firebrick", dash="dash")
        ), row=1, col=col)

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

# Bayesian Init

In [11]:
@njit(fastmath=True, cache=True)
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = (henon_inputs + noise) @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

In [12]:
@njit(fastmath=True, cache=True)
def compute_closed_states(
    steps_start,
    steps_end,
    Y_pred_scaled,
    X_pred,
    W_in,
    W_res,
    bias,
    alpha,
    W_out,
    W_bias,
    res_size,
    scaler_mean,
    scaler_std,
):
    for i in range(steps_start, steps_end):
        u = Y_pred_scaled[i - 2]
        prev_state = X_pred[i - 1, :res_size]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X_pred[i, :res_size] = (1.0 - alpha) * prev_state + alpha * new_state
        X_scaled = (X_pred[i, :res_size] - scaler_mean) / scaler_std
        Y_pred_scaled[i] = W_out @ X_scaled + W_bias
    return Y_pred_scaled

In [13]:
@njit(fastmath=True, cache=True)
def lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon):
    Q_test = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_test = np.zeros(2)
    for i in range(len(Y_test)):
        J = np.array([[-2.0 * a * Y_test[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_test)
        Q_test_raw, R = np.linalg.qr(Z)
        Q_test = np.ascontiguousarray(Q_test_raw)
        lyapunov_sums_test += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_test = lyapunov_sums_test / total_steps_after_henon

    Q_pred = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_pred = np.zeros(2)
    for i in range(len(Y_pred)):
        J = np.array([[-2.0 * a * Y_pred[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_pred)
        Q_pred_raw, R = np.linalg.qr(Z)
        Q_pred = np.ascontiguousarray(Q_pred_raw)
        lyapunov_sums_pred += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_pred = lyapunov_sums_pred / total_steps_after_henon

    le_loss = (lyapunov_exponent_test - lyapunov_exponent_pred) ** 2
    return le_loss[0]

In [14]:
def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    noise_val,
    tau_steps,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-bias_scaling, bias_scaling, res_size)
    W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
    W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
    mask = rng.random((res_size, res_size)) < sparsity
    W_res *= mask
    try:
        eigenvalues = la.eigvals(W_res)
    except Exception as e:
        print("Eigenval Issue")
        raise optuna.exceptions.TrialPruned()
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res = W_res * (spec_rad / largest_eigenvalue)

    rng = np.random.default_rng(42)
    noise = rng.normal(
        0, noise_val, size=(len(henon_scaled) - test_steps - tau_steps, out_size)
    )
    X = compute_states(
        steps + transient_steps_reservoir - test_steps,
        henon_scaled[: -test_steps - tau_steps],
        res_size,
        W_in,
        W_res,
        bias,
        alpha,
        noise,
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[transient_steps_reservoir:])
    Y_train = henon_scaled[transient_steps_reservoir + tau_steps : -test_steps]
    scaler_mean = scaler.mean_
    scaler_std = scaler.scale_

    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=Warning)
        try:
            model = Ridge(alpha=ridge_alpha, solver="auto")
            model.fit(X_train, Y_train)
        except (Warning, ValueError):
            try:
                model = Ridge(alpha=ridge_alpha, solver="svd")
                model.fit(X_train, Y_train)
            except Exception as e:
                print("Ridge nor SVD worked")
                raise optuna.exceptions.TrialPruned()

    W_out = model.coef_
    W_bias = model.intercept_

    X_pred = np.zeros((test_steps, res_size))
    Y_pred_scaled = np.zeros((test_steps, out_size))
    Y_test = henon_dataset[-test_steps:]

    u = Y_train[-2]
    prev_state = X_train[-1]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[0] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[0].reshape(1, -1))
    Y_pred_scaled[0] = model.predict(X_pred_scaled)

    u = Y_train[-1]
    prev_state = X_pred[0]
    new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
    X_pred[1] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[1].reshape(1, -1))
    Y_pred_scaled[1] = model.predict(X_pred_scaled)

    compute_closed_states(
        0,
        test_steps,
        Y_pred_scaled,
        X_pred,
        W_in,
        W_res,
        bias,
        alpha,
        W_out,
        W_bias,
        res_size,
        scaler_mean,
        scaler_std,
    )

    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)

    return Y_test, Y_pred

In [15]:
def normalize_2d(data):
    min_vals = data.min(axis=0)
    max_vals = data.max(axis=0)
    return (data - min_vals) / (max_vals - min_vals)

# RSME

In [98]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    rmse = root_mean_squared_error(Y_test, Y_pred)

    return rmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #199...

In [100]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 193: [0.5972473103342589]
Params: {'sparsity': 0.30052987906811496, 'spec_rad': 1.8151521173180747, 'alpha': 0.6349518319558759, 'input_scaling': 0.8255349578816455, 'bias_scaling': 0.1613730036139721, 'ridge_alpha': 3.422685011407857e-12, 'noise_val': 0.07862994454054789}
R^2: -0.7501325305884832, MSE: 0.5972473103342589


# RSME smaller input and bias

In [102]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-3, 0.3, log=True),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-3, 0.3, log=True),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    rmse = root_mean_squared_error(Y_test, Y_pred)

    return rmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #0...

[Optuna] Processing Trial #199...

In [103]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 181: [0.4695207032204268]
Params: {'sparsity': 0.40312307343931947, 'spec_rad': 1.643188367681495, 'alpha': 0.7760962384599176, 'input_scaling': 0.13497756084113774, 'bias_scaling': 0.0678038835810532, 'ridge_alpha': 0.0001658072759806235, 'noise_val': 7.803464511580869e-05}
R^2: -0.0023905422201492277, MSE: 0.4695207032204268


# RSME first couple points

In [104]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    rmse = root_mean_squared_error(Y_test[:30], Y_pred[:30])

    return rmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #199...

In [105]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 151: [0.44507900975955517]
Params: {'sparsity': 0.06237198166958888, 'spec_rad': 1.9456175793031865, 'alpha': 0.672261863990388, 'input_scaling': 0.05623319224717324, 'bias_scaling': 0.06885238222874057, 'ridge_alpha': 0.0002858762642667265, 'noise_val': 2.496254199139962e-05}
R^2: -0.05858717710030048, MSE: 0.48322989480867645


# NRME

In [107]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return nrmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #199...

In [108]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 150: [0.8705706993051076]
Params: {'sparsity': 0.49091995987361425, 'spec_rad': 1.9986686001970857, 'alpha': 0.7995599644092507, 'input_scaling': 0.14847836769031586, 'bias_scaling': 0.15370284864074318, 'ridge_alpha': 2.4043128917165397e-13, 'noise_val': 3.913670036970602e-06}
R^2: -0.008033762662190025, MSE: 0.47040514205384143


# NRME first couple points

In [109]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    rmse = root_mean_squared_error(Y_test[:30], Y_pred[:30])
    std = np.std(Y_test[:30])
    nrmse = rmse / std

    return nrmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #1...

[Optuna] Processing Trial #199...

In [110]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 110: [0.8389267482487055]
Params: {'sparsity': 0.10266572060708669, 'spec_rad': 1.8132190327976916, 'alpha': 0.7258310944433176, 'input_scaling': 0.17271382248780098, 'bias_scaling': 0.33148669634823225, 'ridge_alpha': 5.398975657739299e-07, 'noise_val': 0.07073643266029508}
R^2: -0.030116598553671547, MSE: 0.4745295954228499


# Lypanauv

In [111]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    return lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon)


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #199...

In [113]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 111: [2.459394931100058e-08]
Params: {'sparsity': 0.23336917582232625, 'spec_rad': 1.9584550091473234, 'alpha': 0.5245849402932763, 'input_scaling': 1.4395363920063833, 'bias_scaling': 0.24180569430856286, 'ridge_alpha': 0.000278281547537853, 'noise_val': 0.031761974781900584}
R^2: -2.116716093764456, MSE: 0.777295089086882


# Hausdorff

In [116]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    return max(h_uv, h_vu)


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #199...

In [117]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 4: [0.3016588079872984]
Params: {'sparsity': 0.313515443836993, 'spec_rad': 1.4176138249474592, 'alpha': 0.286025785861984, 'input_scaling': 1.834608388004151, 'bias_scaling': 0.29203747328681723, 'ridge_alpha': 2.8171799978087537e-08, 'noise_val': 1.7018334722377005e-05}
R^2: -37.63903877959173, MSE: 3.2649591937970315


# Hausdorff with Drift

In [127]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    return max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #199...

In [132]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 143: [1.3207192366655907, 0.3370586526808903]
Params: {'sparsity': 0.3312759874043521, 'spec_rad': 1.935754166905168, 'alpha': 0.5962621530509499, 'input_scaling': 0.8475644970445474, 'bias_scaling': 0.44432432855575577, 'ridge_alpha': 3.198293025877243e-05, 'noise_val': 1.0170367752511206e-06}
R^2: -1.9642908507862527, MSE: 0.7136389044942824


Trial 145: [0.8741164692509135, 0.4046719069731518]
Params: {'sparsity': 0.3243748868058227, 'spec_rad': 1.9643810299009454, 'alpha': 0.678027980058245, 'input_scaling': 0.11263203863011956, 'bias_scaling': 0.9745137012979589, 'ridge_alpha': 7.071342035461124e-13, 'noise_val': 2.9516183915800326e-06}
R^2: -0.01642132821635267, MSE: 0.4723210673386901


Trial 146: [5.999640496073968, 0.31708646685099845]
Params: {'sparsity': 0.2225361003482717, 'spec_rad': 1.6267943531923441, 'alpha': 0.2062420036762434, 'input_scaling': 1.0472437514654305, 'bias_scaling': 0.58880092421307, 'ridge_alpha': 0.00014570358405881242, 'noise_val': 0.0012087619993737166}
R^2: -50.767944071336935, MSE: 3.241852433214664


# Hausdorff with Drift vs NRME

In [130]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)
    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return nrmse, lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon)


study = optuna.create_study(directions=["minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #2...

[Optuna] Processing Trial #199...

In [131]:
for trial  in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 143: [1.3207192366655907, 0.3370586526808903]
Params: {'sparsity': 0.3312759874043521, 'spec_rad': 1.935754166905168, 'alpha': 0.5962621530509499, 'input_scaling': 0.8475644970445474, 'bias_scaling': 0.44432432855575577, 'ridge_alpha': 3.198293025877243e-05, 'noise_val': 1.0170367752511206e-06}
R^2: -1.9642908507862527, MSE: 0.7136389044942824


Trial 145: [0.8741164692509135, 0.4046719069731518]
Params: {'sparsity': 0.3243748868058227, 'spec_rad': 1.9643810299009454, 'alpha': 0.678027980058245, 'input_scaling': 0.11263203863011956, 'bias_scaling': 0.9745137012979589, 'ridge_alpha': 7.071342035461124e-13, 'noise_val': 2.9516183915800326e-06}
R^2: -0.01642132821635267, MSE: 0.4723210673386901


Trial 146: [5.999640496073968, 0.31708646685099845]
Params: {'sparsity': 0.2225361003482717, 'spec_rad': 1.6267943531923441, 'alpha': 0.2062420036762434, 'input_scaling': 1.0472437514654305, 'bias_scaling': 0.58880092421307, 'ridge_alpha': 0.00014570358405881242, 'noise_val': 0.0012087619993737166}
R^2: -50.767944071336935, MSE: 3.241852433214664


# Lypanauv vs NRME

In [135]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return nrmse, lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon)


study = optuna.create_study(directions=["minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #199...

In [136]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 165: [1.4861935381222338, 0.00013498326968088557]
Params: {'sparsity': 0.1311556493639333, 'spec_rad': 1.8071323458786477, 'alpha': 0.665057315907788, 'input_scaling': 1.823521329974827, 'bias_scaling': 1.022753549354906, 'ridge_alpha': 3.0433219664608735e-09, 'noise_val': 0.00011360538183140003}
R^2: -1.896741590351852, MSE: 0.8030514729911367


Trial 170: [1.6129167589399978, 9.004047573833782e-06]
Params: {'sparsity': 0.1311556493639333, 'spec_rad': 1.6085903317241874, 'alpha': 0.7313570359257512, 'input_scaling': 0.7902697966534901, 'bias_scaling': 0.13510074412577064, 'ridge_alpha': 8.634637467853977e-10, 'noise_val': 0.00012524591478911703}
R^2: -3.275177314047468, MSE: 0.8715252393812558


# Lupanuv vs RSME vs Haussdorf

In [16]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        rmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[I 2026-06-26 23:49:29,701] A new study created in memory with name: no-name-da7a48b1-c867-49cd-995b-3acd6695e68c


[Optuna] Processing Trial #199...

In [17]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 53: [5.41006596817727, 1.2078423511601661, 0.00013313289339925347]
Params: {'sparsity': 0.4531443161450162, 'spec_rad': 1.78904584708649, 'alpha': 0.5345983611972178, 'input_scaling': 0.13642071810391357, 'bias_scaling': 0.24994744329791527, 'ridge_alpha': 1.3597084423474808e-08, 'noise_val': 0.05677873890702686}
R^2: -6.514717148326895, MSE: 1.2078423511601661


Trial 145: [5.406288783191977, 0.8003371558562443, 0.00032610521207572166]
Params: {'sparsity': 0.4157178643862894, 'spec_rad': 1.4170727147669369, 'alpha': 0.6423790202518773, 'input_scaling': 1.5857711841304143, 'bias_scaling': 0.7971267525787123, 'ridge_alpha': 2.9593841995773453e-13, 'noise_val': 0.03635831025423842}
R^2: -1.77141483096355, MSE: 0.8003371558562443


Trial 146: [0.43018689360137585, 1.1468804374555714, 0.007995935798837391]
Params: {'sparsity': 0.26658302579695436, 'spec_rad': 1.7570966601363167, 'alpha': 0.36506432690063423, 'input_scaling': 1.779669067227306, 'bias_scaling': 0.4263353552496832, 'ridge_alpha': 1.1631652436257941e-10, 'noise_val': 1.5267593720505716e-06}
R^2: -4.668353236253589, MSE: 1.1468804374555714


Trial 168: [0.39684413730867324, 0.9955085781840122, 0.008506431083084902]
Params: {'sparsity': 0.07865497494947003, 'spec_rad': 1.4868983352134029, 'alpha': 0.47398470708506674, 'input_scaling': 1.9984889946821645, 'bias_scaling': 0.1239590133710767, 'ridge_alpha': 4.396277351250574e-09, 'noise_val': 0.06145343697171091}
R^2: -3.0065396571063063, MSE: 0.9955085781840122


Trial 177: [5.379606584785073, 0.9789578351256998, 0.00312253160273642]
Params: {'sparsity': 0.21685400886921993, 'spec_rad': 1.7649764527822283, 'alpha': 0.7731553622376631, 'input_scaling': 1.5857711841304143, 'bias_scaling': 0.5620418614275343, 'ridge_alpha': 2.1651450533627358e-13, 'noise_val': 0.007710146918716514}
R^2: -2.7156729930713888, MSE: 0.9789578351256998


Trial 185: [0.3700164588790263, 1.1170358780058858, 0.011343680095846943]
Params: {'sparsity': 0.07865497494947003, 'spec_rad': 1.3839811918431326, 'alpha': 0.47398470708506674, 'input_scaling': 1.2443681939375577, 'bias_scaling': 0.17374961831835678, 'ridge_alpha': 4.396277351250574e-09, 'noise_val': 0.06145343697171091}
R^2: -3.6713872407070838, MSE: 1.1170358780058858


Trial 188: [5.380395094887283, 0.8318047631964008, 0.002138286325780094]
Params: {'sparsity': 0.28863473907082116, 'spec_rad': 1.7649764527822283, 'alpha': 0.6780509683952965, 'input_scaling': 1.2443681939375577, 'bias_scaling': 0.479587707581002, 'ridge_alpha': 0.000964989262498755, 'noise_val': 2.1170410447600575e-05}
R^2: -1.889417584973544, MSE: 0.8318047631964008


Trial 189: [0.33088302472314157, 1.8105997866338288, 0.11761955081052405]
Params: {'sparsity': 0.07865497494947003, 'spec_rad': 1.6305153516943574, 'alpha': 0.5944059044600204, 'input_scaling': 1.779669067227306, 'bias_scaling': 0.4263353552496832, 'ridge_alpha': 4.396277351250574e-09, 'noise_val': 6.003765020397679e-05}
R^2: -10.617143695136072, MSE: 1.8105997866338288


Trial 190: [0.33945226856190874, 1.2857259305456745, 0.02548473080471399]
Params: {'sparsity': 0.0842977531869031, 'spec_rad': 1.5164215914775903, 'alpha': 0.6528310866389918, 'input_scaling': 1.4145842171223468, 'bias_scaling': 0.4263353552496832, 'ridge_alpha': 0.000964989262498755, 'noise_val': 1.8981735730470682e-06}
R^2: -5.260061570329736, MSE: 1.2857259305456745


# Lupanuv vs NRSME vs Haussdorf

In [25]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #199...

In [27]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=200,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 20: [10.383779192748117, 1.2643365244578515, 5.147630851629511e-05]
Params: {'sparsity': 0.1934830615374498, 'spec_rad': 1.679810931855063, 'alpha': 0.6822065625911883, 'input_scaling': 1.889561566143881, 'bias_scaling': 1.667190463394921, 'ridge_alpha': 0.00012706301165121064, 'noise_val': 0.05415638684271028}
R^2: -0.9986134127318592, MSE: 0.6831730069323348


Trial 25: [0.43787942582617895, 1.0083528244198672, 0.008600180354757615]
Params: {'sparsity': 0.28505782250292333, 'spec_rad': 1.940838765626423, 'alpha': 0.7402833549863609, 'input_scaling': 0.6185608810756386, 'bias_scaling': 0.282388408518454, 'ridge_alpha': 8.712290721477748e-10, 'noise_val': 0.0954250588309874}
R^2: -0.4241588368610344, MSE: 0.5448544891187299


Trial 146: [0.3729083898446178, 1.4383565652849257, 4.3527597683413604e-05]
Params: {'sparsity': 0.1934830615374498, 'spec_rad': 1.940838765626423, 'alpha': 0.7402833549863609, 'input_scaling': 1.9987917641488502, 'bias_scaling': 1.801954895602728, 'ridge_alpha': 0.00012706301165121064, 'noise_val': 5.267862647761119e-05}
R^2: -1.7641402870144947, MSE: 0.7772031897662115


Trial 153: [10.387479698978826, 1.2410436592773164, 2.5395802211584338e-05]
Params: {'sparsity': 0.1934830615374498, 'spec_rad': 1.679810931855063, 'alpha': 0.6822065625911883, 'input_scaling': 1.889561566143881, 'bias_scaling': 1.5989364745961894, 'ridge_alpha': 1.742836224708996e-05, 'noise_val': 0.05415638684271028}
R^2: -0.9252487989471191, MSE: 0.6705869141970331


Trial 158: [0.3207822026847195, 2.688128615078371, 0.026205945999580097]
Params: {'sparsity': 0.1648045151388035, 'spec_rad': 1.9693263629442799, 'alpha': 0.2911756010084535, 'input_scaling': 1.5328308442976566, 'bias_scaling': 0.20878292094729337, 'ridge_alpha': 8.550313743408413e-12, 'noise_val': 4.6214606231005264e-05}
R^2: -9.61874010223601, MSE: 1.4525064122239273


Trial 184: [0.31775740440148875, 2.309267372736356, 0.0329734418858695]
Params: {'sparsity': 0.1457736392318384, 'spec_rad': 1.8675797433080294, 'alpha': 0.3286868287309134, 'input_scaling': 1.5541037852286312, 'bias_scaling': 0.4580764202150605, 'ridge_alpha': 1.6409024022209712e-11, 'noise_val': 0.05903727815909787}
R^2: -6.459849261583603, MSE: 1.2477921062349424


Trial 197: [5.423735739836563, 1.4041087693627123, 2.0306663246547173e-05]
Params: {'sparsity': 0.2837584896217245, 'spec_rad': 1.6924758716087576, 'alpha': 0.7204170568779975, 'input_scaling': 1.5541037852286312, 'bias_scaling': 0.20878292094729337, 'ridge_alpha': 8.550313743408413e-12, 'noise_val': 4.6214606231005264e-05}
R^2: -1.4366031440400537, MSE: 0.7586976975429158


Trial 199: [0.4107791508246951, 1.4168390398266921, 0.0010227966328685922]
Params: {'sparsity': 0.2837584896217245, 'spec_rad': 1.4933555213031788, 'alpha': 0.6822065625911883, 'input_scaling': 1.889561566143881, 'bias_scaling': 1.369411582092983, 'ridge_alpha': 1.742836224708996e-05, 'noise_val': 0.05415638684271028}
R^2: -1.490607356823188, MSE: 0.7655763860753603


In [28]:
params = study.best_trials[3].params
Y_test, Y_pred = henon_closed(
    in_size=2,
    out_size=2,
    res_size=200,
    sparsity=params["sparsity"],
    spec_rad=params["spec_rad"],
    alpha=params["alpha"],
    input_scaling=params["input_scaling"],
    bias_scaling=params["bias_scaling"],
    ridge_alpha=params["ridge_alpha"],
    noise_val=params["noise_val"],
    tau_steps=1,
)
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"Trial {trial.number}: {trial.values}")
print(f"Params: {trial.params}")
print(f"R^2: {r_2}, MSE: {mse}")
henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 199: [0.4107791508246951, 1.4168390398266921, 0.0010227966328685922]
Params: {'sparsity': 0.2837584896217245, 'spec_rad': 1.4933555213031788, 'alpha': 0.6822065625911883, 'input_scaling': 1.889561566143881, 'bias_scaling': 1.369411582092983, 'ridge_alpha': 1.742836224708996e-05, 'noise_val': 0.05415638684271028}
R^2: -0.9252487989471191, MSE: 0.6705869141970331


In [30]:
params = study.best_trials[3].params
print(f"Trial {trial.number}: {trial.values}")
print(f"Params: {trial.params}")
print()
for res_size in [5, 10, 25, 50, 100, 200, 500, 1000, 2000]:
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=res_size,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Res_size: {res_size}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 199: [0.4107791508246951, 1.4168390398266921, 0.0010227966328685922]
Params: {'sparsity': 0.2837584896217245, 'spec_rad': 1.4933555213031788, 'alpha': 0.6822065625911883, 'input_scaling': 1.889561566143881, 'bias_scaling': 1.369411582092983, 'ridge_alpha': 1.742836224708996e-05, 'noise_val': 0.05415638684271028}

Res_size: 5
R^2: -5.201005332376761, MSE: 1.1361451994787832


Res_size: 10
R^2: -973.9815449927033, MSE: 16.064086922093313


Res_size: 25
R^2: -181.61009168146458, MSE: 2.6805882439886126


Res_size: 50
R^2: -87.81120783709957, MSE: 5.009262115688674


Res_size: 100
R^2: -8.454807644764452, MSE: 1.3800732652715089


Res_size: 200
R^2: -0.9252487989471191, MSE: 0.6705869141970331


Res_size: 500
R^2: -36.65265931559319, MSE: 3.150749215741883


Res_size: 1000
R^2: -3.6406978856449452, MSE: 1.129460997368291


Res_size: 2000
R^2: -5.654216195428929, MSE: 1.3715089919623957


# Lupanuv vs NRSME vs Haussdorf for res of 500

In [23]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=500,
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, 0.1, log=True),
        tau_steps=1,
    )

    drift_penalty_x = np.sum(np.abs(Y_pred[0]) > 1.5) * 5
    drift_penalty_y = np.sum(np.abs(Y_pred[1]) > 0.6) * 5

    Y_test_norm = normalize_2d(Y_test)
    Y_pred_norm = normalize_2d(Y_pred)

    h_uv = directed_hausdorff(Y_test_norm, Y_pred_norm)[0]
    h_vu = directed_hausdorff(Y_pred_norm, Y_test_norm)[0]

    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return (
        max(h_uv, h_vu) + drift_penalty_x + drift_penalty_y,
        nrmse,
        lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon),
    )


study = optuna.create_study(directions=["minimize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=200, n_jobs=-1)

[Optuna] Processing Trial #199...

In [24]:
for trial in study.best_trials:
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=500,
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=params["noise_val"],
        tau_steps=1,
    )
    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)
    print(f"Trial {trial.number}: {trial.values}")
    print(f"Params: {trial.params}")
    print(f"R^2: {r_2}, MSE: {mse}")
    henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()

Trial 44: [0.4286355680884862, 0.8943381212492646, 0.006084328785846454]
Params: {'sparsity': 0.26990338160314964, 'spec_rad': 1.7473966728374315, 'alpha': 0.6038349211156927, 'input_scaling': 0.06763947343258783, 'bias_scaling': 0.34467024764495197, 'ridge_alpha': 4.1470869047160526e-13, 'noise_val': 1.162827946268411e-05}
R^2: -0.06810676118668402, MSE: 0.48324765731976865


Trial 79: [0.42095965318403067, 1.1748756796253539, 0.002222304903080114]
Params: {'sparsity': 0.4575601264733906, 'spec_rad': 1.7473966728374315, 'alpha': 0.6038349211156927, 'input_scaling': 0.5937670950920031, 'bias_scaling': 0.12103146063755328, 'ridge_alpha': 0.00018329922611868716, 'noise_val': 0.02683585898134312}
R^2: -0.7870097286781317, MSE: 0.6348336343170167


Trial 127: [0.39279214261526596, 0.9923765235609404, 0.0056492400169629526]
Params: {'sparsity': 0.278678596181988, 'spec_rad': 1.9854664453092334, 'alpha': 0.43150352720610635, 'input_scaling': 0.4689097873908288, 'bias_scaling': 0.07316939247993808, 'ridge_alpha': 2.403580925887301e-13, 'noise_val': 0.0026280775014957273}
R^2: -0.29493756067769805, MSE: 0.5362218369044559


Trial 148: [0.4254950054358868, 1.3824543142995203, 8.320947235791942e-10]
Params: {'sparsity': 0.46091720805368497, 'spec_rad': 1.9373843288371653, 'alpha': 0.7599198047731797, 'input_scaling': 1.3377274783198232, 'bias_scaling': 0.5436327533705961, 'ridge_alpha': 1.3202621342901557e-13, 'noise_val': 3.604425716921955e-05}
R^2: -1.1949091859585521, MSE: 0.7469969051566913


Trial 149: [0.40794912918933596, 1.1034976319601149, 0.004861719239800999]
Params: {'sparsity': 0.2659368676359284, 'spec_rad': 1.8517579057086337, 'alpha': 0.5632478666880119, 'input_scaling': 0.6085942393591037, 'bias_scaling': 0.023395534129300517, 'ridge_alpha': 3.3479094808464796e-12, 'noise_val': 9.15335728312248e-06}
R^2: -0.6895920245377243, MSE: 0.5962651404792461


Trial 165: [0.3354271186103671, 4.436833794127706, 0.13877224767525081]
Params: {'sparsity': 0.3938312707321368, 'spec_rad': 1.854183992538436, 'alpha': 0.6304183788316424, 'input_scaling': 1.9984024900745454, 'bias_scaling': 0.2526881979819748, 'ridge_alpha': 8.146203947442689e-06, 'noise_val': 0.0026280775014957273}
R^2: -19.500889101048674, MSE: 2.397403717885136


Trial 190: [0.380859385570632, 1.3379031258157468, 2.318843158314343e-06]
Params: {'sparsity': 0.3755666207325404, 'spec_rad': 1.9373843288371653, 'alpha': 0.7599198047731797, 'input_scaling': 1.3377274783198232, 'bias_scaling': 0.4511536210320342, 'ridge_alpha': 0.00014968331918089134, 'noise_val': 1.9796164654195465e-05}
R^2: -1.140879413802665, MSE: 0.722924066311891


Trial 192: [0.4145506716506925, 0.9283576528511762, 0.009136014329161609]
Params: {'sparsity': 0.39664077531649566, 'spec_rad': 1.9854664453092334, 'alpha': 0.43150352720610635, 'input_scaling': 0.4689097873908288, 'bias_scaling': 0.07316939247993808, 'ridge_alpha': 0.0004898639897951213, 'noise_val': 1.0029658495234867e-05}
R^2: -0.15705047691441665, MSE: 0.5016298089457951
